In [2]:
import pandas as pd
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

In [14]:
tickers = ["SPY", "^VIX"]
raw = yf.download(tickers, start="2000-01-01", end="2023-12-31", interval="1mo", auto_adjust=True)
raw.head()

[*********************100%***********************]  2 of 2 completed


Price           Close                  High                   Low             \
Ticker            SPY       ^VIX        SPY       ^VIX        SPY       ^VIX   
Date                                                                           
2000-01-01  87.676735  24.950001  93.134445  29.000000  84.810456  19.510000   
2000-02-01  86.341713  23.370001  90.817818  28.120001  83.377275  20.690001   
2000-03-01  94.469376  24.110001  97.846086  25.870001  84.830044  19.190001   
2000-04-01  91.383270  26.200001  96.431689  34.310001  84.081269  23.250000   
2000-05-01  89.946465  23.650000  93.518737  32.889999  85.970713  23.230000   

Price            Open                Volume       
Ticker            SPY       ^VIX        SPY ^VIX  
Date                                              
2000-01-01  93.134445  24.360001  156770800    0  
2000-02-01  87.794484  24.430000  186938300    0  
2000-03-01  86.459503  22.650000  247594900    0  
2000-04-01  94.552063  24.990000  229246200    0  
2000-05-01  92.308297  26.070000  161024000    0

In [15]:
# flatten multi index cols
raw.columns = ["_".join(col).strip() for col in raw.columns]
df = raw[['Close_SPY', 'Close_^VIX']].copy()
df.columns turn = ['SPY', 'VIX']


df.dropna(inplace=True)
print(df.shape)
print(df.head(10))

(288, 2)
                  SPY        VIX
Date                            
2000-01-01  87.676735  24.950001
2000-02-01  86.341713  23.370001
2000-03-01  94.469376  24.110001
2000-04-01  91.383270  26.200001
2000-05-01  89.946465  23.650000
2000-06-01  91.501350  19.540001
2000-07-01  90.276604  20.740000
2000-08-01  96.175362  16.840000
2000-09-01  90.671173  20.570000
2000-10-01  90.473724  23.629999


In [16]:
df["SPY_ret"] = df["SPY"].pct_change()
df["SPY_lag1"] = df["SPY_ret"].shift(1)


# crises dummy
df["crisis"] = 0
df.loc["2008-09-01":"2009-03-31", "crisis"] = 1
df.loc["2020-02-01":"2020-05-31", "crisis"] = 1

df.dropna(inplace=True)

In [17]:
df.head()

,SPY,VIX,SPY_ret,SPY_lag1,crisis
Date,,,,,
2000-03-01,94.469376,24.110001,0.094134,-0.015227,0
2000-04-01,91.383270,26.200001,-0.032668,0.094134,0
2000-05-01,89.946465,23.650000,-0.015723,-0.032668,0
2000-06-01,91.501350,19.540001,0.017287,-0.015723,0
2000-07-01,90.276604,20.740000,-0.013385,0.017287,0


In [8]:
# Monthly returns for spy
close["SPY_ret"] = close["SPY"].pct_change()
close["VIX_level"] = close["VIX"]
close.dropna(inplace=True)

In [18]:
close.head()

,SPY,VIX
Date,,
2000-01-01,87.676735,24.950001
2000-02-01,86.341713,23.370001
2000-03-01,94.469376,24.110001
2000-04-01,91.383270,26.200001
2000-05-01,89.946465,23.650000


In [23]:
# Fetch Fama-French 3 factors
ff = pd.read_csv(
    "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip",
    skiprows=3,
    index_col=0
)

# Keep only monthly rows (format is YYYYMM, 6 characters)
ff = ff[ff.index.astype(str).str.strip().str.len() == 6]

# Convert index to datetime
ff.index = pd.to_datetime(ff.index.astype(str).str.strip(), format='%Y%m') + pd.offsets.MonthEnd(0)

# Clean column names
ff.columns = ff.columns.str.strip()

# Convert from percentage to decimal
ff = ff[['Mkt-RF', 'SMB', 'HML', 'RF']].astype(float) / 100

print(ff.shape)
print(ff.tail(5))

(1198, 4)
            Mkt-RF     SMB     HML      RF
2025-12-31 -0.0036 -0.0103  0.0236  0.0034
2026-01-31  0.0103  0.0212  0.0386  0.0030
2026-02-28 -0.0117  0.0024  0.0265  0.0028
2026-03-31 -0.0518  0.0044  0.0335  0.0029
2026-04-30  0.0994  0.0013 -0.0127  0.0029


In [25]:
# Align SPY index to month-end so it matches FF
df.index = df.index + pd.offsets.MonthEnd(0)

# Merge
df_ff = df.join(ff, how='inner')
df_ff.dropna(inplace=True)

# Compute SPY excess return (SPY return minus risk-free rate)
df_ff['SPY_excess'] = df_ff['SPY_ret'] - df_ff['RF']

print(df_ff.shape)
print(df_ff[['SPY_ret', 'SPY_excess', 'Mkt-RF', 'SMB', 'HML']].tail(5))

(286, 10)
             SPY_ret  SPY_excess  Mkt-RF     SMB     HML
2023-08-31 -0.016252   -0.020752 -0.0236 -0.0304 -0.0129
2023-09-30 -0.050783   -0.055083 -0.0523 -0.0247  0.0147
2023-10-31 -0.018258   -0.022958 -0.0315 -0.0386  0.0021
2023-11-30  0.091344    0.086944  0.0888 -0.0014  0.0178
2023-12-31  0.041433    0.037133  0.0485  0.0628  0.0494


In [27]:
Y = df["SPY_ret"]
x = df[["VIX", "SPY_lag1", "crisis"]]

X = sm.add_constant(x)
model = sm.OLS(Y, X).fit()

In [22]:
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                SPY_ret   R-squared:                       0.208
Model:                            OLS   Adj. R-squared:                  0.202
Method:                 Least Squares   F-statistic:                     37.18
Date:                Tue, 02 Jun 2026   Prob (F-statistic):           4.60e-15
Time:                        14:49:42   Log-Likelihood:                 513.55
No. Observations:                 286   AIC:                            -1021.
Df Residuals:                     283   BIC:                            -1010.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0624      0.007      9.079      0.000       0.049       0.076
VIX           -0.0027      0.000     -8.623      0.000      -0.003      -0.002
SPY_lag1      -0.1575      0.056     -2.815      0.005      -0.268      -0.047
==============================================================================
Omnibus:                       30.528   Durbin-Watson:                   1.608
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               40.085
Skew:                           0.745   Prob(JB):                     1.98e-09
Kurtosis:                       4.068   Cond. No.                         508.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Fama-French regression

In [30]:
# Define X and Y
Y = df_ff['SPY_excess']
X = df_ff[['Mkt-RF', 'SMB', 'HML']]

# Add constant (this captures alpha)
X = sm.add_constant(X)

# Fit
ff_model = sm.OLS(Y, X).fit()

print(ff_model.summary())

                            OLS Regression Results                            
Dep. Variable:             SPY_excess   R-squared:                       0.986
Model:                            OLS   Adj. R-squared:                  0.986
Method:                 Least Squares   F-statistic:                     6482.
Date:                Tue, 02 Jun 2026   Prob (F-statistic):          1.00e-259
Time:                        15:00:11   Log-Likelihood:                 1086.8
No. Observations:                 286   AIC:                            -2166.
Df Residuals:                     282   BIC:                            -2151.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -1.822e-05      0.000     -0.056      0.9